In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pre_processing import pre_process, json_decode


sns.set_palette("hsv_r")

# This affects things like the size of the labels, lines, and other elements of the plot, but not the overall style. 
# The base context is “notebook”, and the other contexts are “paper”, “talk”, and “poster”, which are version of 
# the notebook parameters scaled by .8, 1.3, and 1.6, respectively.
sns.set_context("talk")

%config Inline.figure_format = 'retina'
%matplotlib inline

# new 

## Read in Data 

In [3]:
# File paths for each experimental condition (can contain multiple subjects/files) 
## WITHOUT FB 
file_paths_baseline = [
    "../data/csv_files/01_baseline_withoutFB.csv",   # Abby1 60
    "../data/csv_files/02_baseline_withoutFB.csv",     # Yannie 
    "../data/csv_files/03_baseline_withoutFB.csv",   # Eric1 60 
    "../data/csv_files/04_baseline_withoutFB.csv",     # Paul
    "../data/csv_files/05_baseline_withoutFB.csv",     # Jason 60
]

file_paths_repulse = [
    "../data/csv_files/01_repulse_withoutFB.csv",    # Abby1
    "../data/csv_files/02_repulse_withoutFB.csv",      # Yannie 
    "../data/csv_files/03_repulse_withoutFB.csv",    # Eric1
    "../data/csv_files/04_repulse_withoutFB.csv",      # Paul
    "../data/csv_files/05_repulse_withoutFB.csv",      # Jason
]

file_paths_udl = [
    "../data/csv_files/01_udl_withoutFB.csv",        # Abby1
    "../data/csv_files/02_udl_withoutFB.csv",          # Yannie
    "../data/csv_files/03_udl_withoutFB.csv",        # Eric1
    "../data/csv_files/04_udl_withoutFB.csv",          # Paul
    "../data/csv_files/05_udl_withoutFB.csv",          # Jason

]

# Subject identifiers corresponding to the datasets
subject_list = ["abby1", "yannie", "eric1", "paul", "jason"]

# Load CSV files & label condition
df_baseline_raw = pre_process(file_paths_baseline, subject_list)
df_baseline_raw['phase'] = 'baseline'

df_repulse_raw = pre_process(file_paths_repulse, subject_list)
df_repulse_raw['phase'] = 'repulse'

df_udl_raw = pre_process(file_paths_udl, subject_list)
df_udl_raw['phase'] = 'udl'

# Combine all conditions into a single dataset
df_withoutFB = pd.concat([df_baseline_raw, df_repulse_raw, df_udl_raw]).reset_index(drop=True)

# Reset the trial number column that resets for each subject
df_withoutFB["TN"] = df_withoutFB.groupby("SN").cumcount()


# File paths for each experimental condition (can contain multiple subjects/files) 
## WITH FB 
file_paths_baseline = [
    "../data/csv_files/01_baseline_withFB.csv",        # Eric2  60
    "../data/csv_files/02_baseline_withFB.csv",      # Abby2
    "../data/csv_files/03_baseline_withFB.csv",        # Rainie 60
    "../data/csv_files/04_baseline_withFB.csv",        # Alice 
    "../data/csv_files/05_baseline_withFB.csv"         # Rhys 60
]

file_paths_repulse = [
    "../data/csv_files/01_repulse_withFB.csv",         # Eric2 
    "../data/csv_files/02_repulse_withFB.csv",       # Abby2
    "../data/csv_files/03_repulse_withFB.csv",         # Rainie 
    "../data/csv_files/04_repulse_withFB.csv",         # Alice 
    "../data/csv_files/05_repulse_withFB.csv"          # Rhys
]

file_paths_udl = [
    "../data/csv_files/01_udl_withFB.csv",             # Eric2 
    "../data/csv_files/02_udl_withFB.csv",           # Abby2
    "../data/csv_files/03_udl_withFB.csv",             # Rainie 
    "../data/csv_files/04_udl_withFB.csv",             # Alice 
    "../data/csv_files/05_udl_withFB.csv"              # Rhys
]

# Subject identifiers corresponding to the datasets
subject_list = ["eric2", "abby2", "rainie", "alice", "rhys"]

# Load CSV files & label condition
df_baseline_raw = pre_process(file_paths_baseline, subject_list)
df_baseline_raw['phase'] = 'baseline'

df_repulse_raw = pre_process(file_paths_repulse, subject_list)
df_repulse_raw['phase'] = 'repulse'

df_udl_raw = pre_process(file_paths_udl, subject_list)
df_udl_raw['phase'] = 'udl'

# Combine all conditions into a single dataset
df_withFB = pd.concat([df_baseline_raw, df_repulse_raw, df_udl_raw]).reset_index(drop=True)

# Reset the trial number column that resets for each subject
df_withFB["TN"] = df_withFB.groupby("SN").cumcount()

## Define Functions 

In [4]:
# function to unpack repeated target value from series
# WE HAVENT USE THIS ONE YET.. copied from Jeremy's analysis script
def find_rep(group):
    vals = group['Repeated_Target'].iloc[0]
    return vals[1:4]   

#function returns df where N-1 is a single target, 
#to compare the repulsive and attractive effect of a single target
def N_finder(group, N_minus_targ):
    N_trial_num = group.loc[(group['target_angle'] == N_minus_targ),'TN'] + 1
    N_df = group.loc[(group['TN'].isin(N_trial_num))]
    return N_df

def add_bias_correction(df, angle_col, phase_name, epoch):
    """
    calcylate target-specific bias and corrected angle for a given phase and angle column
    """
    bias_col = f"bias_{phase_name}_{angle_col}"
    corrected_col = f"corrected_{phase_name}_{angle_col}"

    bias = (
        df.loc[
            (df["TN"] >= epoch[0]) &
            (df["TN"] <= epoch[1]) &
            (~df["mt"].isna()) &
            (df["rt"] > 0.5) &
            (df[angle_col].abs() < 60)
        ]
        .groupby(["SN", "target_angle"], as_index=False)[angle_col]
        .mean()
        .rename(columns={angle_col: bias_col})
    )

    # merge bias back into dataframe
    df = df.merge(bias, on=["SN", "target_angle"], how="left")

    # wrap the angle
    diff = np.deg2rad(df[angle_col] - df[bias_col])

    df[corrected_col] = np.rad2deg(
        np.arctan2(np.sin(diff), np.cos(diff))
    )
    return df

def calculate_se_index(df, y_col):
    """
    calculate the sequential effect (SE) index per participant 
    POSITIVE SE Index = attractive bias (UDL)
    NEGATIVE SE Index = repulsive bias (REPULSE)

    parameters: 
    df : DataFrame; data to calculate SE index from
    y_col : str; column containing corrected bias

    output: 
    df with the SE index per participant 
    """
    negative_mean = df.loc[df["target_angle_diff"] < 0].groupby("SN")[y_col].mean().copy()
    positive_mean = df.loc[df["target_angle_diff"] > 0].groupby("SN")[y_col].mean().copy()

    return negative_mean - positive_mean 

## Data Wrangling 

In [5]:
## Define epochs for each phase of the experiment (trial number ranges for each phase)
epochs = {
    'baseline':(0, 15),      # Baseline reaching
    'repulse':(16, 715),   # Main experiment block
    'repulse_clean':(116, 715),   # For bias correction
    'udl_withoutFB':(716, 1579),   
    'udl_baseline_FB': (716, 785),      # Baseline reaching 
    'udl_baseline_noFB': (786, 855),    # For UDL bias corrrection  
    'udl_withFB':(856, 1719),                  # Main experiment block
    
}

In [ ]:

## running add_bias_correction for all angle calculations & 
angle_cols = ["theta_pv"]
phase_info = {"repulse": epochs["repulse_clean"]}

for phase_name, epoch in phase_info.items():
    for angle in angle_cols:
        df_withoutFB = add_bias_correction(
            df_withoutFB,
            angle_col=angle,
            phase_name=phase_name,
            epoch=epoch
        )

angle_cols = ["theta_pv"]
phase_info = {
    "repulse": epochs["repulse_clean"],
    "udl": epochs["udl_baseline_noFB"]
}

for phase_name, epoch in phase_info.items():
    for angle in angle_cols:
        df_withFB = add_bias_correction(
            df_withFB,
            angle_col=angle,
            phase_name=phase_name,
            epoch=epoch
        )

## Shift the target angles for participants who had a central target of 60 degrees, so that central target is always at 150 degrees 
# Participants who had central target as 60 degrees
p_list1 = [1, 3, 5, 6, 8, 10]

# Shift all targets so central target is always 150 degrees
mask = df['SN'].isin(set(p_list1))
TA_rad_shift = np.deg2rad(df.loc[mask]['target_angle']) + np.deg2rad(90)
df.loc[mask,'target_angle'] = (np.rad2deg(np.arctan2(np.sin(TA_rad_shift), np.cos(TA_rad_shift))) % 360).round()

# Get target location difference values
df['target_angle_diff'] = pd.Series(pd.NA, index=df.index, dtype='Float64')
# NOTES:
# 1) CW is negative, CCW is positive 
# 2) add one to starting index because we want the first trial for every block to be NA
mask = (df['TN'] >= epochs['repulse'][0]) & (df['TN'] <= epochs['repulse'][1])
RadTAN1diff_repulse = np.deg2rad(df.loc[mask].groupby('SN')['target_angle'].diff(periods=1)) 
df.loc[mask,'target_angle_diff'] = np.rad2deg(np.arctan2(np.sin(RadTAN1diff_repulse), np.cos(RadTAN1diff_repulse)))

# Round 
df['target_angle_diff'] = df['target_angle_diff'].round().astype('Int64')